[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/rolling_ball/rolling_ball.ipynb)

# The Rolling Ball

This example computes the orientation and contact trail of a ball rolling without slipping around a closed loop on a flat table. Each small displacement turns the ball in the vertical plane of its motion by an angle proportional to the distance rolled. Because turns in different vertical planes do not commute, rolling once around a closed loop brings the ball back to its starting point with a net turn, which for a small loop approaches a turn about the vertical equal to the enclosed area divided by the square of the ball's radius. Meanwhile, the moving point of contact traces a continuous curve on the ball's surface.

Positions and displacements on the table are modelled as intrinsic antivectors, so joining the upward direction with each displacement `upward & steps` forms the vertical plane patch of the motion and its dual `(upward & steps).dual()` gives the bivector generator of the step rotor `((upward & steps).dual() * (-1 / (2 * radius))).exp()`. Accumulating those step rotors with `step_turns.cumprod(axis=0)` tracks the ball's orientation, while pulling the downward offset back into the body frame via `orientation << (-radius * upward)` records the contact trail on the sphere. Joining successive points around a circuit in `(small_path[:-1] & small_path[1:]).sum() / 2` gives the enclosed area as an intrinsic table patch whose dual generates the small-loop rotor `(small_area.dual() / (2 * radius**2)).exp()`.

In matrix rigid-body notation, rolling without slipping defines the angular velocity $\boldsymbol\omega = (\hat{\mathbf z} \times \dot{\mathbf x}) / r$, integrates the $3 \times 3$ rotation matrix through the skew-symmetric matrix product $\dot R = [\boldsymbol\omega]_\times R$, pulls the contact point into the body frame as $-r R^\top \hat{\mathbf z}$, and writes the small-loop rotation angle about $\hat{\mathbf z}$ via the signed area integral $\frac{1}{2 r^2} \oint (x\,dy - y\,dx)$.

In [ ]:
# Install numga and its examples when running in Colab.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext
from numga.algebras import VGA3D
from examples.util.animation import show_animation
from examples.mechanics.rolling_ball import render

ga = VGA3D
mv = NumpyContext(ga).multivector
# Positions, displacements, and sphere offsets are intrinsic antivectors.
Point = ga.gatype.antivector()                      # Antivector-
Rotor = ga.gatype.rotor()                           # Rotor+
upward = mv.z.dual()                                # [] Point-
radius = 1.0

## Rolling along a circular loop

A rotor in the horizontal plane sweeps a starting point on the first axis around a circular loop on the table. Each displacement multiplies the ball's orientation from the left by a no-slip step rotor in the vertical plane of motion, while pulling the downward radius back through that orientation traces the contact point in the body frame.

In [ ]:
loop_radius = 2.5
n_loops = 3
n_steps_per_loop = 720
n_steps = n_loops * n_steps_per_loop
n_frames_per_loop = 56
n_frames = n_loops * n_frames_per_loop
frame_duration_ms = 40

In [ ]:
def rolling_turns(path: Point) -> Rotor:
    """Cumulative multiplication of the no-slip step rotors along `path` (`[...] Point-`) gives the orientation trajectory (`[...] Rotor+`)."""
    steps = path[1:] - path[:-1]                                                # [...] Point-
    # Dualizing the vertical motion patch gives the bivector axis of the step turn.
    step_turns = ((upward & steps).dual() * (-1 / (2 * radius))).exp()          # [...] Rotor+
    return step_turns.cumprod(axis=0)


def rolling_states(
    path: Point,
    orientation: Rotor,
    contact: Point,
    n_frames: int,
) -> Iterator[tuple[Point, Rotor, Point]]:
    """Yields the ball centre, orientation, and contact trail so far across `n_frames` evenly spaced steps."""
    for step in np.linspace(0, len(orientation) - 1, n_frames).astype(int):
        yield path[step + 1], orientation[step], contact[:step + 1]


angles = np.linspace(0.0, 2 * np.pi * n_loops, n_steps + 1)                     # [n_steps + 1]
circle_rotors = (mv.xy * (-angles / 2)).exp()                                   # [n_steps + 1] Rotor+
path = circle_rotors >> (loop_radius * mv.x.dual())                             # [n_steps + 1] Point-
orientation = rolling_turns(path)                                               # [n_steps] Rotor+
contact = orientation << (-radius * upward)                                     # [n_steps] Point-

show_animation(render.rolling_frames(path, rolling_states(path, orientation, contact, n_frames), radius), "rolling_ball", frame_duration_ms)

Gold and blue octants separated by dark great-circle seams mark the ball's body frame, while the red curve traces the contact points accumulated over three circuits of the grey table loop. Because each circuit leaves a net turn, the contact curve does not close after one revolution around the table and instead precesses across the sphere from lap to lap.

## Net turn around one loop

After one full circuit around the loop, the accumulated step rotors leave a net turn rather than returning to the identity. When the loop radius is small compared with the ball's radius, that net turn approaches a horizontal rotation generated by the dual of the enclosed table area divided by twice the squared radius.

In [ ]:
small_loop_radius = 0.05 * radius

In [ ]:
loop_path = path[:n_steps_per_loop + 1]                                         # [n_steps_per_loop + 1] Point-
loop_turn = orientation[n_steps_per_loop - 1]                                   # [] Rotor+
loop_area = (loop_path[:-1] & loop_path[1:]).sum() / 2                          # [] Antibivector-

small_path = circle_rotors[:n_steps_per_loop + 1] >> (small_loop_radius * mv.x.dual())  # [n_steps_per_loop + 1] Point-
small_turn = rolling_turns(small_path)[-1]                                      # [] Rotor+
small_area = (small_path[:-1] & small_path[1:]).sum() / 2                       # [] Antibivector-
area_turn = (small_area.dual() / (2 * radius**2)).exp()                         # [] Rotor+

print("turn after one large loop:")
print(loop_turn.coefficient_table())
print("enclosed area of the large loop:")
print(loop_area.coefficient_table())
print("turn after one small loop vs. area-law rotor:")
print(small_turn.coefficient_table())
print(area_turn.coefficient_table())

In [ ]:
# checks
# Step rotors keep the orientation normalized, the contact trail stays on the sphere, the join sum recovers
# the disc area, and the small-loop holonomy matches the area-law rotor.
assert ((orientation.symmetric_reverse_product() - 1).abs() < 1e-9).all()
assert ((contact.scalar_norm_squared() - radius**2).abs() < 1e-9).all()
assert ((loop_area - np.pi * loop_radius**2 * (mv.x.dual() & mv.y.dual())).norm() < 1e-3).all()
assert ((small_turn - area_turn).norm() < 3e-4).all()